# 1. Load and audit the PPIP data

This notebook downloads the official PPIP OCDS snapshot and turns each compiled contracting process into one row. It also checks missing values, duplicate identifiers and date coverage before modelling.

Run this notebook first. The source file is kept in `data/raw/`; the process-level table is written to `data/processed/`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gzip
import hashlib
import json
import os
import shutil
import urllib.request

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path.cwd().resolve()
if not (PROJECT / "notebooks").exists():
    PROJECT = PROJECT.parent
RAW_DIR = PROJECT / "data" / "raw"
PROCESSED_DIR = PROJECT / "data" / "processed"
REPORT_DIR = PROJECT / "reports"
for folder in (RAW_DIR, PROCESSED_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

REGISTRY_URL = "https://data.open-contracting.org/en/publication/147"
DOWNLOAD_URL = os.getenv(
    "PPIP_DOWNLOAD_URL",
    "https://fastly.data.open-contracting.org/downloads/kenya_ppra/4361/full.jsonl.gz",
)
ARCHIVE = RAW_DIR / "ppip_ocds_all_time.jsonl.gz"
PROCESS_FILE = PROCESSED_DIR / "ppip_processes.csv"
print("Project folder:", PROJECT)
print("Source page:", REGISTRY_URL)

## Download the source file

The registry provides one compiled record per contracting process in compressed JSONL format. The download is only done when the file is not already present. Check the registry page before reusing an old snapshot.

In [ ]:
if not ARCHIVE.exists() or ARCHIVE.stat().st_size == 0:
    request = urllib.request.Request(
        DOWNLOAD_URL,
        headers={"User-Agent": "DSA8401-PPIP-Capstone/1.0"},
    )
    temporary = ARCHIVE.with_suffix(ARCHIVE.suffix + ".part")
    try:
        with urllib.request.urlopen(request, timeout=180) as response, temporary.open("wb") as target:
            shutil.copyfileobj(response, target, length=1024 * 1024)
        with temporary.open("rb") as source:
            if source.read(2) != b"\x1f\x8b":
                raise ValueError("The downloaded file is not a gzip archive. Check PPIP_DOWNLOAD_URL.")
        temporary.replace(ARCHIVE)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
else:
    print("Using the source file already in data/raw.")

sha = hashlib.sha256()
with ARCHIVE.open("rb") as source:
    for chunk in iter(lambda: source.read(1024 * 1024), b""):
        sha.update(chunk)
source_sha256 = sha.hexdigest()
print(f"Archive size: {ARCHIVE.stat().st_size / (1024 * 1024):.1f} MB")
print("SHA-256:", source_sha256)

## Read the OCDS records

The loader accepts a direct compiled release and the common wrapper formats. Award suppliers are kept only for data-quality checks; they are not treated as the set of competing bidders.

In [ ]:
def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value] if isinstance(value, dict) else []


def get_compiled_release(payload):
    if isinstance(payload.get("compiledRelease"), dict):
        return payload["compiledRelease"]
    record = payload.get("record")
    if isinstance(record, dict):
        return record.get("compiledRelease", record)
    releases = payload.get("releases")
    if isinstance(releases, list) and releases:
        tender_releases = [item for item in releases if isinstance(item, dict) and item.get("tender")]
        return sorted(tender_releases, key=lambda item: str(item.get("date", "")))[-1] if tender_releases else releases[0]
    return payload


def to_number(value):
    try:
        number = float(value)
        return number if np.isfinite(number) else np.nan
    except (TypeError, ValueError):
        return np.nan


def to_date(value):
    return pd.to_datetime(value, utc=True, errors="coerce") if value else pd.NaT


def extract_process(payload):
    release = get_compiled_release(payload)
    tender = release.get("tender") or {}
    buyer = release.get("buyer") or {}
    parties = as_list(release.get("parties"))
    if not buyer:
        buyer = next(
            (party for party in parties if "buyer" in [str(role).lower() for role in as_list(party.get("roles"))]),
            {},
        )

    address = buyer.get("address") or {}
    period = tender.get("tenderPeriod") or {}
    value = tender.get("value") or {}
    tenderers = as_list(tender.get("tenderers"))
    if not tenderers:
        tenderers = [
            party for party in parties
            if "tenderer" in [str(role).lower() for role in as_list(party.get("roles"))]
        ]

    amount = to_number(value.get("amount"))
    currency = str(value.get("currency") or "").upper()
    start = to_date(period.get("startDate"))
    end = to_date(period.get("endDate"))
    duration = np.nan
    if pd.notna(start) and pd.notna(end):
        days = (end - start).total_seconds() / 86400
        if 0 < days <= 365:
            duration = round(days, 2)

    title = str(tender.get("title") or "").strip()
    description = str(tender.get("description") or "").strip()
    text = " ".join(part for part in (title, description) if part)
    identifier = buyer.get("identifier") or {}

    return {
        "ocid": str(release.get("ocid") or "").strip(),
        "record_date": to_date(release.get("date")),
        "tender_start_date": start,
        "tender_end_date": end,
        "buyer_name": str(buyer.get("name") or "").strip(),
        "buyer_identifier": str(identifier.get("id") or buyer.get("id") or ""),
        "county_raw": str(address.get("region") or address.get("locality") or "").strip(),
        "procurement_method": str(tender.get("procurementMethod") or "unknown").strip().lower(),
        "procurement_category": str(tender.get("mainProcurementCategory") or "unknown").strip().lower(),
        "estimated_value_kes": amount if currency == "KES" else np.nan,
        "reported_currency": currency or "unknown",
        "duration_days": duration,
        "num_tenderers": len(tenderers) if tenderers else np.nan,
        "tenderer_data_available": int(bool(tenderers)),
        "tender_title": title,
        "tender_text": text,
        "award_count": len(as_list(release.get("awards"))),
        "contract_count": len(as_list(release.get("contracts"))),
    }

rows = []
invalid_lines = 0
with gzip.open(ARCHIVE, "rt", encoding="utf-8") as stream:
    for line in stream:
        if not line.strip():
            continue
        try:
            row = extract_process(json.loads(line))
            if row["ocid"]:
                rows.append(row)
        except (json.JSONDecodeError, TypeError, ValueError, AttributeError):
            invalid_lines += 1

processes = pd.DataFrame(rows)
if processes.empty:
    raise ValueError("No contracting processes were read. Check the archive format.")
processes["record_date"] = pd.to_datetime(processes["record_date"], utc=True, errors="coerce")
processes["tender_start_date"] = pd.to_datetime(processes["tender_start_date"], utc=True, errors="coerce")
processes["tender_end_date"] = pd.to_datetime(processes["tender_end_date"], utc=True, errors="coerce")
processes = processes.sort_values(["record_date", "ocid"], na_position="first")
duplicate_rows = int(processes.duplicated("ocid", keep="last").sum())
processes = processes.drop_duplicates("ocid", keep="last").reset_index(drop=True)
processes.to_csv(PROCESS_FILE, index=False)

print("Unique procurement processes:", f"{len(processes):,}")
print("Duplicate OCIDs removed:", f"{duplicate_rows:,}")
print("Invalid JSONL lines:", f"{invalid_lines:,}")
display(processes.head().T)

## Data-quality summary

The audit is written as a small CSV table so that the important checks are easy to read and compare between snapshots.

In [ ]:
checks = [
    ("unique_processes", len(processes)),
    ("duplicate_ocid_rows_removed", duplicate_rows),
    ("invalid_jsonl_lines", invalid_lines),
    ("tender_start_date_missing_pct", round(processes["tender_start_date"].isna().mean() * 100, 2)),
    ("buyer_name_missing_pct", round(processes["buyer_name"].fillna("").eq("").mean() * 100, 2)),
    ("buyer_identifier_missing_pct", round(processes["buyer_identifier"].fillna("").eq("").mean() * 100, 2)),
    ("value_missing_or_non_kes_pct", round(processes["estimated_value_kes"].isna().mean() * 100, 2)),
    ("duration_missing_pct", round(processes["duration_days"].isna().mean() * 100, 2)),
    ("tenderers_not_disclosed_pct", round(processes["num_tenderers"].isna().mean() * 100, 2)),
    ("tender_text_missing_pct", round(processes["tender_text"].fillna("").eq("").mean() * 100, 2)),
    ("award_array_empty_pct", round(processes["award_count"].eq(0).mean() * 100, 2)),
    ("contract_array_empty_pct", round(processes["contract_count"].eq(0).mean() * 100, 2)),
    ("first_tender_start_date", processes["tender_start_date"].min()),
    ("last_tender_start_date", processes["tender_start_date"].max()),
    ("archive_sha256", source_sha256),
    ("retrieved_at_utc", datetime.now(timezone.utc).isoformat()),
]
audit = pd.DataFrame(checks, columns=["check", "value"])
audit.to_csv(REPORT_DIR / "data_audit.csv", index=False)
(PROJECT / "data" / "processed" / "source_sha256.txt").write_text(source_sha256 + "\n", encoding="utf-8")
display(audit)
print("Saved:", PROCESS_FILE)